# 11: Observabilidade com o LangSmith

Numa análise, o pipeline faz dezenas de chamadas: LLM do Supervisor, agentes, tools, buscas no Google. Quando
algo sai estranho no relatório, a pergunta é "em qual passo?". Um **trace** responde: é o registro em árvore de
uma execução, com cada passo, a entrada, a saída, o tempo e os tokens gastos.

Com `LANGSMITH_TRACING=true` no `.env`, o LangChain e o LangGraph mandam os traces sozinhos, sem código extra.
A busca na web usa o SDK do Google direto (fora do LangChain), por isso o cliente é envolvido com `wrap_gemini`:
ela aparece no trace como o passo `busca_google`.

In [1]:
import sys
import warnings
from pathlib import Path

# Sobe pastas até achar a raiz do projeto (onde está o pyproject.toml), de onde quer que o notebook seja aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ))

from dotenv import load_dotenv
_ = load_dotenv(RAIZ / ".env")

# Aviso de depreciação interno do LangGraph, sem relação com o projeto: só polui a saída.
# O filtro vem depois do import do langchain_core, que instala o filtro dele ao ser importado.
from langchain_core._api.deprecation import LangChainPendingDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainPendingDeprecationWarning)
warnings.filterwarnings("ignore", message="Function wrap_gemini is in beta")

## 1. As variáveis estão no `.env`?

Conferimos sem imprimir os valores. Se faltar alguma, o resto do notebook explica e não roda.

In [2]:
import os

NECESSARIAS = ("LANGSMITH_TRACING", "LANGSMITH_API_KEY", "LANGSMITH_AGENT_ID", "LANGSMITH_AGENT_ENVIRONMENT")
for nome in NECESSARIAS:
    print(f"{nome}: {'definida' if os.getenv(nome) else 'FALTANDO'}")

TEM_LANGSMITH = os.getenv("LANGSMITH_TRACING", "").lower() == "true" and bool(os.getenv("LANGSMITH_API_KEY"))
print("\nTracing ligado:", TEM_LANGSMITH)
if not TEM_LANGSMITH:
    print("Sem tracing: preencha as variáveis no .env (veja o .env.example) e rode o notebook de novo.")

LANGSMITH_TRACING: definida
LANGSMITH_API_KEY: definida
LANGSMITH_AGENT_ID: definida
LANGSMITH_AGENT_ENVIRONMENT: definida

Tracing ligado: True


O workspace do LangSmith é organizado **por agente**: na interface, escolha o agente do `LANGSMITH_AGENT_ID`
(`pipeline-b3`) e o ambiente do `LANGSMITH_AGENT_ENVIRONMENT` (`local`). Nunca combine com `LANGSMITH_PROJECT`:
com os dois, o SDK deixa de enviar.

## 2. Duas chamadas pequenas, traçadas

Uma pergunta ao LLM e uma busca na web, dentro de uma função `@traceable` (ela vira a raiz da árvore no trace).
Guardamos o id do trace para lê-lo de volta.

In [3]:
from langsmith import traceable
from langsmith.run_helpers import get_current_run_tree

from fluxo_financeiro.shared.llm import criar_llm
from fluxo_financeiro.tools.web_search import pesquisar_na_web

ids = {}


@traceable(name="notebook_12_exemplo")
def exemplo():
    ids["raiz"] = get_current_run_tree().id
    resposta = criar_llm().invoke("Em uma frase: o que é dividend yield?").text
    busca = pesquisar_na_web("Em uma frase: qual a última notícia sobre a WEG (WEGE3)?")
    return {"llm": resposta, "busca": busca}


if TEM_LANGSMITH:
    saida = exemplo()
    print("LLM:", saida["llm"])
    print("Busca:", saida["busca"][:300])

LLM: O *dividend yield* é um indicador financeiro que mostra a porcentagem de retorno que uma empresa pagou aos seus acionistas em forma de dividendos em relação ao preço atual de suas ações.
Busca: A WEG (WEGE3) anunciou a aquisição de um terreno de 400 mil metros quadrados na Índia, por cerca de US$ 8 milhões, com o objetivo de ampliar a capacidade produtiva e expandir suas operações industriais a longo prazo no país.

Fontes: advfn.com


## 3. Lendo o trace de volta

O envio é assíncrono: o SDK manda os dados em segundo plano, então tentamos ler por alguns segundos.
Imprimimos a árvore com o tipo de cada passo, o tempo e os tokens.

In [4]:
import time
import warnings

from langsmith import Client

# read_run() funciona até jan/2027; a substituta (client.runs) é assíncrona e exige servidor >= 0.16.
# Para um notebook de estudo, a leitura síncrona é mais clara, então o aviso de depreciação fica silenciado.
warnings.filterwarnings("ignore", message=r"read_run\(\) is deprecated")


def ler_trace(run_id, tentativas=15):
    client = Client()
    for _ in range(tentativas):
        try:
            run = client.read_run(run_id, load_child_runs=True)
            if run.end_time:
                return run
        except Exception:
            pass
        time.sleep(2)
    return None


def mostrar(run, nivel=0):
    duracao = (run.end_time - run.start_time).total_seconds() if run.end_time else 0
    tokens = f", {run.total_tokens} tokens" if run.total_tokens else ""
    print(f"{'   ' * nivel}{run.name} [{run.run_type}] {duracao:.1f}s{tokens}")
    for filho in sorted(run.child_runs or [], key=lambda r: r.start_time):
        mostrar(filho, nivel + 1)


if TEM_LANGSMITH:
    trace = ler_trace(ids["raiz"])
    if trace:
        mostrar(trace)
    else:
        print("O trace ainda não chegou ao LangSmith; abra a interface em alguns segundos.")

notebook_12_exemplo [chain] 3.1s, 134 tokens
   ChatGoogleGenerativeAI [llm] 1.1s, 52 tokens
   busca_google [llm] 1.6s, 82 tokens


## O que observar

- A raiz `notebook_12_exemplo` tem dois filhos: a chamada do LLM (`ChatGoogleGenerativeAI`) e a busca (`busca_google`, o nome dado pelo `wrap_gemini`). Os tokens da raiz são a soma dos filhos.
- Cada passo tem tempo e tokens. Numa análise completa da VALE3 são cerca de 29,5 mil tokens e US$ 0,016.
- O custo que o LangSmith calcula cobre tokens; a taxa por busca do Google Search fica de fora.
- Nos testes, o `tests/conftest.py` desliga o tracing: sem isso, cada teste viraria um trace no projeto real.